# Plot ImageNet Debug Curves

This notebook loads the five `summary.csv` files under `files/debug/` and plots `FID` and `IS` versus the number of sampling steps.

In [16]:
from pathlib import Path
import csv

import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 20,
    "axes.titlesize": 24,
    "axes.labelsize": 24,
    "axes.grid": False,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 26,
})

In [ ]:
repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

summary_paths = {
    "analytic": repo_root / "files/debug/fid5k_plain_dit_native_velocity_analytic_imagenet/summary.csv",
    "data": repo_root / "files/debug/fid5k_plain_dit_native_velocity_data_imagenet/summary.csv",
    "ddpm": repo_root / "files/debug/fid5k_plain_dit_psample_imagenet/summary.csv",
    "velocity_aligned": repo_root / "files/debug/fid5k_dit_transport_velocity_aligned_steps/summary.csv",
    "velocity_noalign": repo_root / "files/debug/fid5k_dit_transport_velocity_noalign_steps/summary.csv",
}

for name, path in summary_paths.items():
    print(f"{name}: {path}")
    print("  exists:", path.exists())

In [ ]:
def load_summary(csv_path):
    rows = []
    with csv_path.open(newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                "num_steps": int(row["num_steps"]),
                "fid": float(row["fid"]),
                "is": float(row["is"]),
            })
    return sorted(rows, key=lambda item: item["num_steps"])

series = {name: load_summary(path) for name, path in summary_paths.items()}
series

In [ ]:
colors = {
    "analytic": "#ff6b6b",
    "data": "#ffd12a",
    "ddpm": "#66bb3a",
    "velocity_aligned": "#2c7fb8",
    "velocity_noalign": "#8c564b",
}

labels = {
    "analytic": "Analytic",
    "data": "Data",
    "ddpm": "DDPM",
    "velocity_aligned": "Velocity aligned",
    "velocity_noalign": "Velocity noalign",
}

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for name, rows in series.items():
    steps = [row["num_steps"] for row in rows]
    fid_values = [row["fid"] for row in rows]
    is_values = [row["is"] for row in rows]

    axes[0].plot(steps, fid_values, marker="o", markersize=9, linewidth=3, color=colors[name], label=labels[name])
    axes[1].plot(steps, is_values, marker="o", markersize=9, linewidth=3, color=colors[name], label=labels[name])

axes[0].set_xlabel("Number of steps", fontweight="bold")
axes[0].set_ylabel("FID", fontweight="bold")
axes[0].set_xscale("log", base=2)

axes[1].set_xlabel("Number of steps", fontweight="bold")
axes[1].set_ylabel("Inception Score", fontweight="bold")
axes[1].set_xscale("log", base=2)

for ax in axes:
    ax.legend(frameon=False)
    ax.set_xticks([1, 2, 4, 8, 16, 32, 64, 250])
    ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Optional: print the best FID from each method
for name, rows in series.items():
    best = min(rows, key=lambda row: row["fid"])
    print(f"{labels[name]}: best FID = {best['fid']:.4f} at {best['num_steps']} steps")